# Phase 1 — EDA, Cleaning, Churn Labels, Time Split

**Goal of this notebook**
1. Load Online Retail II and profile it (shape, dates, missing, return rate).
2. Apply cleaning rules (drop nulls / returns / non-product codes).
3. Build three rolling 90-day windows: `train`, `val`, `test`.
4. Generate churn labels per window (1 = no purchase in next 90 days).
5. Persist processed transactions and churn label tables to `data/processed/`.

All logic that we'll reuse later (loader, cleaner, splits, labels) lives in `src/data/`. The notebook is a thin narrative on top — when something proves out, it gets moved into a module.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

from src.data.loader import load_raw, clean
from src.data.splits import build_windows, customers_active_before
from src.data.labels import make_churn_labels

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "online_retail_II.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH, PROCESSED_DIR

## 1. Load and profile raw data

In [ ]:
raw = load_raw(RAW_PATH)
print(f"Rows: {len(raw):,}")
print(f"Columns: {list(raw.columns)}")
print(f"Date range: {raw['invoice_date'].min()} to {raw['invoice_date'].max()}")
raw.head()

In [ ]:
raw.info()
raw.isna().sum().to_frame("missing")

In [ ]:
# Returns (negative quantity) — important: spec says drop these for churn modeling
returns_mask = raw["quantity"] <= 0
neg_price_mask = raw["price"] <= 0
null_cust_mask = raw["customer_id"].isna()

print(f"Returns (qty<=0):      {returns_mask.sum():>8,}  ({returns_mask.mean()*100:.2f}%)")
print(f"Non-positive price:    {neg_price_mask.sum():>8,}  ({neg_price_mask.mean()*100:.2f}%)")
print(f"Null customer_id:      {null_cust_mask.sum():>8,}  ({null_cust_mask.mean()*100:.2f}%)")

## 2. Apply cleaning

In [ ]:
df = clean(raw)
print(f"After clean: {len(df):,} rows  ({len(df)/len(raw)*100:.1f}% of raw)")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Unique products:  {df['stock_code'].nunique():,}")
print(f"Date range:       {df['invoice_date'].min()} to {df['invoice_date'].max()}")
df.head()

## 3. Quick visual EDA

In [ ]:
# Transactions per day
daily = df.groupby(df["invoice_date"].dt.date).size()
fig, ax = plt.subplots(figsize=(12, 3))
daily.plot(ax=ax)
ax.set_title("Transactions per day")
ax.set_ylabel("#rows")
plt.show()

In [ ]:
# Top countries
top_countries = df["country"].value_counts().head(10)
print(top_countries)
print(f"\nUK share: {(df['country']=='United Kingdom').mean()*100:.1f}%")

In [ ]:
# Customer activity distribution
per_customer = df.groupby("customer_id").agg(
    n_orders=("invoice", "nunique"),
    n_items=("quantity", "sum"),
    revenue=("revenue", "sum"),
)
print(per_customer.describe(percentiles=[.5, .9, .95, .99]))

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].hist(np.log1p(per_customer["n_orders"]), bins=40)
axes[0].set_title("log(1+n_orders) per customer")
axes[1].hist(np.log1p(per_customer["revenue"]), bins=40)
axes[1].set_title("log(1+revenue) per customer")
axes[2].hist(np.log1p(per_customer["n_items"]), bins=40)
axes[2].set_title("log(1+n_items) per customer")
plt.tight_layout()
plt.show()

## 4. Build time-based windows

Strategy: 3 rolling 90-day windows ending at the latest date. Each window has a `feature_end` (cutoff for computing features) and a `label_end = feature_end + 90 days` (the observation window for churn label).

```
       train_feat→|◀── 90d ──▶|val_feat→|◀── 90d ──▶|test_feat→|◀── 90d ──▶|end
```

In [ ]:
windows = build_windows(df, horizon_days=90, n_windows=3)
for w in windows:
    print(f"{w.name:>5}  features ≤ {w.feature_end.date()}   label window ({w.feature_end.date()} → {w.label_end.date()})")

## 5. Generate churn labels per window

In [ ]:
label_tables = {w.name: make_churn_labels(df, w) for w in windows}
for name, t in label_tables.items():
    print(f"{name:>5}: {len(t):>6,} customers  |  churn rate = {t['churn'].mean()*100:.2f}%")

In [ ]:
# Sanity: customers in val should largely be subset+superset overlap with train (they accumulate over time).
for name, t in label_tables.items():
    print(name, t.head(3).to_dict(orient="records"))

## 6. Persist processed data

In [ ]:
df.to_parquet(PROCESSED_DIR / "transactions_clean.parquet", index=False)
for name, t in label_tables.items():
    t.to_parquet(PROCESSED_DIR / f"churn_labels_{name}.parquet", index=False)

list(PROCESSED_DIR.glob("*.parquet"))

## 7. Next phase

Phase 2 builds features on top of `transactions_clean.parquet` and joins to each `churn_labels_*.parquet`:
- RFM (recency, frequency, monetary) at each `feature_end`
- Behavioral: avg basket size, purchase interval stats, product diversity
- Time-windowed aggregates (last 7d / 30d / 90d before `feature_end`)
- Target encoding on country / top stock codes

Then train three churn approaches in parallel: GBM stack, BG-NBD, Survival.